In [0]:
df =  spark.read.table('data.orders.mobile_price')
import pyspark.sql.functions as f
from pyspark.sql.window import Window

In [0]:
df.printSchema()

In [0]:
# Get distinct list of Brands
df.select('Brand').distinct().display()

In [0]:
# Find total number of models per Brand. 
model = df.groupBy('Brand').agg(f.count('Model').alias('Total'))
display(model)

In [0]:
# Find top 5 most expensive phones.
top = df.orderBy(f.col('Price_USD').desc()).limit(5)
display(top)

In [0]:
# Get average price per Brand. 
avg_price = df.groupBy('Brand').agg(f.round(f.avg('price_usd'),2).alias('Total'))
display(avg_price)

In [0]:
# Count number of phones by OS. 
OS_phone_count = df.groupBy('os').agg(f.count('Model').alias('Total'))
display(OS_phone_count)

In [0]:
# Find average Camera_MP per Brand. 
avg_cam_mp = df.groupBy('Brand').agg(f.count('camera_mp').alias('Total'))
display(model)

In [0]:
# Find phones with battery > average battery. 
avg_battery = df.select(f.avg('Battery_mAh').alias('avg')).collect()[0]['avg']
filter = df.filter(df['Battery_mAh'] > avg_battery).select("Model", "Battery_mAh")
display(filter)

In [0]:
# Find most expensive phone in each Brand.
w = Window.partitionBy('Brand').orderBy(f.col('Price_USD').desc())
most_expensive = df.withColumn('rank', f.row_number().over(w))\
    .filter(f.col('rank') == 1).select('Brand', 'Model', 'Price_USD')
display(most_expensive)

In [0]:
# Rank phones within each Brand by Price.
w = Window.partitionBy('Brand').orderBy(f.col('Price_USD').desc())
ranked_phones = df.withColumn('rank', f.rank().over(w))\
.select('Brand', 'Model', 'Price_USD', 'rank')
display(ranked_phones)

In [0]:
# Get top 3 brands with highest avg price.
avg_price_brand = df.groupBy('Brand').agg(f.avg('Price_USD').alias('avg_price'))\
    .orderBy(f.col('avg_price').desc()).limit(3)
display(avg_price_brand)

In [0]:
# Create a new column: Price_per_GB = Price_USD / Storage_GB. 
df2 = df.withColumn("Price_per_GB", f.round(f.col('Price_USD') / f.col('Storage_GB'), 2))
display(df2)

In [0]:
# Create a new column with indian price. 
df3 = df.withColumn("Price_INR", f.col('Price_USD')*90)\
    .withColumn("Price_per_GB_INR", f.round(f.col('Price_INR') / f.col('Storage_GB'), 2))
display(df3)